# Random Forest cho phân loại topic từ Clause

Pipeline: **TF-IDF -> TruncatedSVD -> RandomForest**

- x = Clause
- y = code
- Chia dữ liệu: 80 train / 10 valid / 10 test
- Tune hyperparameter bằng RandomizedSearchCV
- Có cơ chế phạt overfit khi chọn mô hình
- Lưu model + best params vào file .pkl


In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    f1_score
)
import joblib

# =========================
# 1) CONFIG
# =========================
DATA_PATH = r"D:\Tài Liệu Môn Học\Thay thế\annotated_clauses_topics.csv"   # đổi lại nếu cần
TEXT_COL = "Clause"
TARGET_COL = "code"
RANDOM_STATE = 42
MODEL_PATH = "random_forest_topic_classifier.pkl"

# RandomizedSearchCV config
N_ITER_SEARCH = 20   # nếu máy yếu có thể giảm xuống 12-15
CV_FOLDS = 3
TOP_K_CANDIDATES = 5

# =========================
# 2) LOAD & CLEAN DATA
# =========================
df = pd.read_csv(DATA_PATH)

# Giữ lại 2 cột cần thiết
df = df[[TEXT_COL, TARGET_COL]].copy()

# Xóa NA
df = df.dropna(subset=[TEXT_COL, TARGET_COL])

# Chuẩn hóa text cơ bản
df[TEXT_COL] = (
    df[TEXT_COL]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Loại bỏ dòng text rỗng
df = df[df[TEXT_COL] != ""].copy()

# Loại bỏ duplicate hoàn toàn giữa (Clause, code) để giảm rò rỉ / overfit
df = df.drop_duplicates(subset=[TEXT_COL, TARGET_COL]).reset_index(drop=True)

print(f"Data shape after cleaning: {df.shape}")
print("\nClass distribution:")
print(df[TARGET_COL].value_counts())

X = df[TEXT_COL]
y = df[TARGET_COL]

# =========================
# 3) SPLIT 80 / 10 / 10
# =========================
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("\nSplit sizes:")
print(f"Train: {len(X_train)} ({len(X_train)/len(df):.1%})")
print(f"Valid: {len(X_valid)} ({len(X_valid)/len(df):.1%})")
print(f"Test : {len(X_test)} ({len(X_test)/len(df):.1%})")

# =========================
# 4) PIPELINE
# =========================
# RF không xử lý text thô trực tiếp tốt, nên dùng:
# TF-IDF -> TruncatedSVD -> RandomForest
base_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("svd", TruncatedSVD(random_state=RANDOM_STATE)),
    ("rf", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

# Search space: ưu tiên các giá trị có xu hướng giảm overfit
param_distributions = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [2, 3, 5],
    "tfidf__max_df": [0.90, 0.95, 1.0],
    "tfidf__sublinear_tf": [True, False],
    "tfidf__max_features": [5000, 10000, 15000, 20000],

    "svd__n_components": [100, 150, 200, 300],

    "rf__n_estimators": [200, 300, 500, 700],
    "rf__max_depth": [10, 20, 30, 40],
    "rf__min_samples_split": [5, 10, 20],
    "rf__min_samples_leaf": [2, 4, 8],
    "rf__max_features": ["sqrt", "log2", 0.5],
    "rf__bootstrap": [True],
    "rf__class_weight": ["balanced", "balanced_subsample"],
    "rf__max_samples": [0.7, 0.85, None]
}

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    estimator=base_pipeline,
    param_distributions=param_distributions,
    n_iter=N_ITER_SEARCH,
    scoring="f1_macro",
    n_jobs=-1,
    cv=cv,
    verbose=2,
    random_state=RANDOM_STATE,
    refit=False,
    return_train_score=True
)

# =========================
# 5) HYPERPARAMETER TUNING ON TRAIN
# =========================
search.fit(X_train, y_train)

cv_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score").reset_index(drop=True)

# Phạt ứng viên có khoảng cách train-CV quá lớn để hạn chế overfit
cv_results["cv_gap"] = cv_results["mean_train_score"] - cv_results["mean_test_score"]
cv_results["overfit_penalized_cv_score"] = (
    cv_results["mean_test_score"] - 0.35 * cv_results["cv_gap"].clip(lower=0)
)

print("\nTop candidates from CV (after overfit penalty):")
display(
    cv_results[
        [
            "params",
            "mean_train_score",
            "mean_test_score",
            "cv_gap",
            "overfit_penalized_cv_score",
            "rank_test_score"
        ]
    ]
    .sort_values(["overfit_penalized_cv_score", "mean_test_score"], ascending=False)
    .head(TOP_K_CANDIDATES)
)

# =========================
# 6) CHỌN MODEL TỐT NHẤT BẰNG VALIDATION
#    - Lấy top K từ CV penalized score
#    - Fit trên train
#    - Chấm trên valid
#    - Phạt train-valid gap để né overfit
# =========================
top_candidates = (
    cv_results
    .sort_values(["overfit_penalized_cv_score", "mean_test_score"], ascending=False)
    .head(TOP_K_CANDIDATES)
    .copy()
)

selection_records = []
best_model = None
best_params = None
best_selection_score = -np.inf

for _, row in top_candidates.iterrows():
    params = row["params"]
    candidate_model = clone(base_pipeline).set_params(**params)
    candidate_model.fit(X_train, y_train)

    train_pred = candidate_model.predict(X_train)
    valid_pred = candidate_model.predict(X_valid)

    train_f1_macro = f1_score(y_train, train_pred, average="macro")
    valid_f1_macro = f1_score(y_valid, valid_pred, average="macro")
    train_valid_gap = train_f1_macro - valid_f1_macro

    # điểm chọn cuối cùng: valid tốt nhưng phạt nếu gap lớn
    selection_score = valid_f1_macro - 0.25 * max(train_valid_gap, 0)

    selection_records.append({
        "params": params,
        "cv_mean_train_f1_macro": row["mean_train_score"],
        "cv_mean_valid_f1_macro": row["mean_test_score"],
        "cv_gap": row["cv_gap"],
        "penalized_cv_score": row["overfit_penalized_cv_score"],
        "train_f1_macro": train_f1_macro,
        "valid_f1_macro": valid_f1_macro,
        "train_valid_gap": train_valid_gap,
        "final_selection_score": selection_score
    })

    if selection_score > best_selection_score:
        best_selection_score = selection_score
        best_model = candidate_model
        best_params = params

selection_df = pd.DataFrame(selection_records).sort_values(
    "final_selection_score", ascending=False
).reset_index(drop=True)

print("\nModel selection on validation set:")
display(selection_df)

print("\nBest params selected:")
print(best_params)

# =========================
# 7) EVALUATION
# =========================
def evaluate_split(model, X_split, y_split, split_name="split"):
    y_pred = model.predict(X_split)

    acc = accuracy_score(y_split, y_pred)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_split, y_pred, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_split, y_pred, average="weighted", zero_division=0
    )

    print(f"\n{'='*80}")
    print(f"{split_name.upper()} METRICS")
    print(f"{'='*80}")
    print(f"Accuracy             : {acc:.4f}")
    print(f"Precision (macro)    : {precision_macro:.4f}")
    print(f"Recall (macro)       : {recall_macro:.4f}")
    print(f"F1-score (macro)     : {f1_macro:.4f}")
    print(f"Precision (weighted) : {precision_weighted:.4f}")
    print(f"Recall (weighted)    : {recall_weighted:.4f}")
    print(f"F1-score (weighted)  : {f1_weighted:.4f}")

    report_dict = classification_report(
        y_split, y_pred, output_dict=True, zero_division=0
    )
    report_df = pd.DataFrame(report_dict).T

    # accuracy của classification_report là 1 dòng scalar
    if "accuracy" in report_df.index:
        report_df.loc["accuracy", ["precision", "recall", "f1-score"]] = acc

    print(f"\nDetailed classification report for {split_name}:")
    display(report_df)

    overall_metrics = pd.DataFrame([{
        "split": split_name,
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted
    }])

    return overall_metrics, report_df

train_overall, train_report = evaluate_split(best_model, X_train, y_train, "train")
valid_overall, valid_report = evaluate_split(best_model, X_valid, y_valid, "valid")
test_overall, test_report = evaluate_split(best_model, X_test, y_test, "test")

all_overall_metrics = pd.concat(
    [train_overall, valid_overall, test_overall],
    ignore_index=True
)

print("\nOverall metrics across splits:")
display(all_overall_metrics)

# =========================
# 8) OVERFITTING CHECK
# =========================
train_f1 = float(train_overall["f1_macro"].iloc[0])
valid_f1 = float(valid_overall["f1_macro"].iloc[0])
test_f1 = float(test_overall["f1_macro"].iloc[0])

print("\nOverfitting check:")
print(f"Train F1_macro - Valid F1_macro = {train_f1 - valid_f1:.4f}")
print(f"Train F1_macro - Test  F1_macro = {train_f1 - test_f1:.4f}")

if (train_f1 - valid_f1 > 0.08) or (train_f1 - test_f1 > 0.08):
    print("WARNING: Mô hình có dấu hiệu overfit. Hãy thử tăng min_samples_leaf, min_samples_split, giảm max_depth, hoặc giảm n_components/max_features.")
else:
    print("Mức chênh lệch train/valid/test hiện tương đối ổn.")

# =========================
# 9) SAVE MODEL + PARAMS TO PKL
# =========================
artifact = {
    "model": best_model,
    "best_params": best_params,
    "text_column": TEXT_COL,
    "target_column": TARGET_COL,
    "classes": sorted(df[TARGET_COL].unique().tolist()),
    "overall_metrics": all_overall_metrics,
    "selection_table": selection_df
}

joblib.dump(artifact, MODEL_PATH)
print(f"\nSaved model artifact to: {MODEL_PATH}")

# =========================
# 10) LOAD AGAIN (OPTIONAL TEST)
# =========================
loaded_artifact = joblib.load(MODEL_PATH)
loaded_model = loaded_artifact["model"]

sample_texts = [
    "The beach was beautiful and relaxing",
    "Parking was difficult and the staff gave poor directions"
]

sample_preds = loaded_model.predict(sample_texts)
print("\nSample predictions after reload:")
for text, pred in zip(sample_texts, sample_preds):
    print(f"- {pred}: {text}")


Data shape after cleaning: (23114, 2)

Class distribution:
code
VE     8989
TC     1871
LC     1718
RA     1696
HL     1477
CT     1475
SE     1405
UH     1101
TCr     982
CA      896
EL      696
EC      460
VR      348
Name: count, dtype: int64

Split sizes:
Train: 18491 (80.0%)
Valid: 2311 (10.0%)
Test : 2312 (10.0%)
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Top candidates from CV (after overfit penalty):


,params,mean_train_score,mean_test_score,cv_gap,overfit_penalized_cv_score,rank_test_score
1,"{'tfidf__sublinear_tf': False, 'tfidf__ngram_r...",0.848899,0.497697,0.351203,0.374776,2
0,"{'tfidf__sublinear_tf': True, 'tfidf__ngram_ra...",0.898854,0.501414,0.397440,0.362310,1
10,"{'tfidf__sublinear_tf': False, 'tfidf__ngram_r...",0.661375,0.424306,0.237070,0.341331,11
8,"{'tfidf__sublinear_tf': True, 'tfidf__ngram_ra...",0.743703,0.437124,0.306579,0.329822,9
9,"{'tfidf__sublinear_tf': True, 'tfidf__ngram_ra...",0.718205,0.426502,0.291703,0.324405,10



Model selection on validation set:


,params,cv_mean_train_f1_macro,cv_mean_valid_f1_macro,cv_gap,penalized_cv_score,train_f1_macro,valid_f1_macro,train_valid_gap,final_selection_score
0,"{'tfidf__sublinear_tf': False, 'tfidf__ngram_r...",0.848899,0.497697,0.351203,0.374776,0.819899,0.524018,0.295881,0.450048
1,"{'tfidf__sublinear_tf': True, 'tfidf__ngram_ra...",0.898854,0.501414,0.397440,0.362310,0.831978,0.505114,0.326864,0.423398
2,"{'tfidf__sublinear_tf': False, 'tfidf__ngram_r...",0.661375,0.424306,0.237070,0.341331,0.612563,0.431478,0.181085,0.386206
3,"{'tfidf__sublinear_tf': True, 'tfidf__ngram_ra...",0.743703,0.437124,0.306579,0.329822,0.669689,0.429069,0.240620,0.368914
4,"{'tfidf__sublinear_tf': True, 'tfidf__ngram_ra...",0.718205,0.426502,0.291703,0.324405,0.663567,0.418304,0.245263,0.356988



Best params selected:
{'tfidf__sublinear_tf': False, 'tfidf__ngram_range': (1, 1), 'tfidf__min_df': 2, 'tfidf__max_features': 20000, 'tfidf__max_df': 0.9, 'svd__n_components': 300, 'rf__n_estimators': 700, 'rf__min_samples_split': 10, 'rf__min_samples_leaf': 8, 'rf__max_samples': 0.7, 'rf__max_features': 0.5, 'rf__max_depth': 20, 'rf__class_weight': 'balanced', 'rf__bootstrap': True}

TRAIN METRICS
Accuracy             : 0.8307
Precision (macro)    : 0.7955
Recall (macro)       : 0.8588
F1-score (macro)     : 0.8199
Precision (weighted) : 0.8385
Recall (weighted)    : 0.8307
F1-score (weighted)  : 0.8316

Detailed classification report for train:


...


TEST METRICS
Accuracy             : 0.5869
Precision (macro)    : 0.5323
Recall (macro)       : 0.5108
F1-score (macro)     : 0.5157
Precision (weighted) : 0.5896
Recall (weighted)    : 0.5869
F1-score (weighted)  : 0.5838

Detailed classification report for test:


,precision,recall,f1-score,support
CA,0.670103,0.730337,0.698925,89.000000
CT,0.572464,0.533784,0.552448,148.000000
EC,0.395349,0.369565,0.382022,46.000000
EL,0.222222,0.376812,0.279570,69.000000
HL,0.586826,0.662162,0.622222,148.000000
LC,0.529851,0.412791,0.464052,172.000000
RA,0.581818,0.564706,0.573134,170.000000
SE,0.511364,0.319149,0.393013,141.000000
TC,0.638158,0.518717,0.572271,187.000000
TCr,0.567568,0.428571,0.488372,98.000000



Overall metrics across splits:


,split,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,train,0.830728,0.795494,0.858763,0.819899,0.838480,0.830728,0.831640
1,valid,0.597577,0.538342,0.520396,0.524018,0.602578,0.597577,0.596280
2,test,0.586938,0.532330,0.510783,0.515662,0.589596,0.586938,0.583754



Overfitting check:
Train F1_macro - Valid F1_macro = 0.2959
Train F1_macro - Test  F1_macro = 0.3042

Saved model artifact to: random_forest_topic_classifier.pkl

Sample predictions after reload:
- CT: The beach was beautiful and relaxing
- SE: Parking was difficult and the staff gave poor directions
